In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error
import xgboost as xgb

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [ ]:
db_path="amazon_analysis.db"
conn=sqlite3.connect(db_path)

query="""
Select
   p.product_id,
   p.name,
   p.brand,
   p.category,
   ph.date,
   ph.price,
   ph.mrp,
   ph.discount_pct,
   ms.reviews,
   ms.rating,
   ms.in_stock
From products p
Join price_history ph on p.product_id=ph.product_id
Join market_signals ms ON p.product_id=ms.product_id
       And ph.date=ms.date
"""
df=pd.read_sql(query,conn)

print(f"data loaded :{len(df)}rows,{len(df.columns)}columns")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique products: {df['product_id'].nunique()}")
print(f"Categories:{df['category'].unique()}")

print("\nFirst 5 rows")
print(df.head())
print("\nNull values:")
print(df.isnull().sum())

In [ ]:
# Check what columns we actually have
print("Columns in dataframe:")
print(df.columns.tolist())

print("\nFirst row:")
print(df.head(1))

print("\nDataframe shape:")
print(df.shape)

In [ ]:
df['date']=pd.to_datetime(df['date'])
#1.day of week mon=0, sun=6
df['day_of_week']=df['date'].dt.dayofweek
#2. days since first date
min_date = df['date'].min()
df['days_since_start'] = (df['date'] - min_date).dt.days
# 3. Price to MRP ratio (price/mrp)
df['price_to_mrp_ratio'] = df['price'] / df['mrp']
df['price_to_mrp_ratio'] = df['price_to_mrp_ratio'].fillna(1)  # Handle divide by zero
# 4. Review velocity (reviews divided by days tracked)
df['review_velocity'] = df['reviews'] / (df['days_since_start'] + 1)
# 5. Encode categorical variables (brand, category as numbers)
le_brand = LabelEncoder()
le_category = LabelEncoder()
df['brand_encoded'] = le_brand.fit_transform(df['brand'])
df['category_encoded'] = le_category.fit_transform(df['category'])

# Handle missing values
df['rating'] = df['rating'].fillna(df['rating'].mean())  # Fill rating nulls with average
df['in_stock'] = df['in_stock'].fillna(1)  # Assume in stock if missing
 
# Show new dataframe
print("\nData with features:")
print(df[['price', 'discount_pct', 'reviews', 'rating', 'day_of_week', 'review_velocity']].head())

In [ ]:
#model1-price predn (xgboost)

print("Question: Given brand, category, ratings, reviews, can we predict price?")
print("Why: Shows you understand pricing patterns\n")
 
X_price = df[['brand_encoded', 'category_encoded', 'rating', 'reviews', 'discount_pct', 'day_of_week']].copy()

y_price = df['price'].copy()

X_train_price, X_test_price, y_train_price, y_test_price = train_test_split(
    X_price, y_price, test_size=0.2, random_state=42
)
 
print(f"Training set: {len(X_train_price)} rows")
print(f"Test set: {len(X_test_price)} rows")

# XGBoost = gradient boosting (builds many simple trees, combines them)
model_price = xgb.XGBRegressor(
    n_estimators=100,      
    max_depth=5,           
    learning_rate=0.1,     
    objective='reg:squarederror'  
)
 
model_price.fit(X_train_price, y_train_price, verbose=0)
 
y_pred_price = model_price.predict(X_test_price)

rmse_price = np.sqrt(mean_squared_error(y_test_price, y_pred_price))
r2_price = r2_score(y_test_price, y_pred_price)
mae_price = mean_absolute_error(y_test_price, y_pred_price)
 
print(f"\nModel Performance:")
print(f"  RMSE (Root Mean Squared Error): ₹{rmse_price:.2f}")
print(f"  R² Score: {r2_price:.4f} (explains {r2_price*100:.2f}% of variance)")
print(f"  MAE (Mean Absolute Error): ₹{mae_price:.2f}")
 
# Example: show actual vs predicted for first 10 test samples
print(f"\nSample Predictions (Actual → Predicted):")
for i in range(min(5, len(y_test_price))):
    print(f"  ₹{y_test_price.iloc[i]:.0f} → ₹{y_pred_price[i]:.0f}")
 
# Feature importance (which features matter most)
feature_importance_price = model_price.feature_importances_
feature_names = ['Brand', 'Category', 'Rating', 'Reviews', 'Discount%', 'Day_of_week']
print(f"\nTop Features (importance):")
for name, importance in sorted(zip(feature_names, feature_importance_price), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {name}: {importance:.4f}")
 

In [ ]:

print("MODEL 2: ANOMALY DETECTION (Fake Discounts)")
 
X_anomaly = df[['price', 'mrp', 'discount_pct', 'reviews', 'rating']].copy()
X_anomaly = X_anomaly.fillna(X_anomaly.mean())
 

model_anomaly = IsolationForest(
    contamination=0.05,  # Expect 5% of data to be anomalies
    random_state=42
)
 
# Fit and predict (-1 = anomaly, 1 = normal)
anomaly_predictions = model_anomaly.fit_predict(X_anomaly)
anomaly_scores = model_anomaly.score_samples(X_anomaly)  # Lower = more anomalous
 

df['anomaly_score'] = anomaly_scores
df['is_anomaly'] = anomaly_predictions
 
# Count anomalies
n_anomalies = (anomaly_predictions == -1).sum()
print(f"Anomalies detected: {n_anomalies} products ({n_anomalies/len(df)*100:.2f}%)")
 
print(f"\nTop 10 Most Anomalous Products:")
anomalies_df = df[df['is_anomaly'] == -1].nlargest(10, 'anomaly_score')
for idx, row in anomalies_df.iterrows():
    print(f"  {row['name'][:50]}...")
    print(f"    Brand: {row['brand']}, Discount: {row['discount_pct']:.1f}%, Price: ₹{row['price']:.0f}")
 
# Evaluation metric: compare with SQL findings
# We found 20 fake discounts in SQL analysis
# How many did our model find?
print(f"\nValidation:")
print(f"  SQL found: 20 fake discounts (high discount, stable price)")
print(f"  Model found: {n_anomalies} anomalies")
print(f"  Recall estimate: {min(n_anomalies, 20) / 20 * 100:.0f}%")

# Count unique products flagged as anomalies
unique_anomalies = df[df['is_anomaly'] == -1]['product_id'].nunique()
print(unique_anomalies)

In [ ]:

print("MODEL EVALUATION SUMMARY")
summary= {
    'Model': ['Price Prediction', 'Anomaly Detection'],
    'Algorithm': ['XGBoost', 'Isolation Forest'],
    'Result': [f'R²={r2_price:.3f}, RMSE=₹{rmse_price:.0f}', f'Caught 124 anomalies'],
  
}
 
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
 

In [ ]:

 
# Save models as pickle files

import joblib
 
joblib.dump(model_price, 'models/price_model.pkl')
joblib.dump(model_anomaly, 'models/anomaly_model.pkl')

 
# save label encoders (needed to convert brand/category to numbers)
joblib.dump(le_brand, 'models/le_brand.pkl')
joblib.dump(le_category, 'models/le_category.pkl')
 
 
conn.close()
